# Classification Matrics: Precision, Recall, Specificity & F1

We already know how to compare models reliably; now we need to answer different questions:

> **What kind of mistakes is the model making, and which mistakes actually matter?**

## 1. Retrieval

1. Suppose a dataset contains $90\%$ class A and $10\%$ class B. A classifier predicts class A for every observation and gets $90\%$ accuracy. What does the dummy-baseline idea tell you about that result?

> A dummy-baseline that uses the most-frequent-class strategy would also get an accuracy of $90\%$ just by predicting class A for every single test observation. This tells us that the classifier performance is no better than how well we could perform without learning any relationships between the features and the target.

2. Why should we avoid using `X_test` to decide which model or preprocessing approach we prefer?

> Anything learned from data must be learned only from the data available to the model at that stage, therefore the test set cannot influence which model is selected or what preprocessing approach is chosen as it should be an independent estimate of how your chosen modelelling process performs on unseen data.

3. In 5-fold cross-validation, how many times is each training observation used for validation and how many times for fitting?

> Each training observation is used four times for fitting and exactly once for validation across the five cross-validation runs.

4. Why did putting `StandardScaler()` inside the k-NN pipeline prevent leakage during cross-validation?

> Putting `StandardScaler()` inside the pipeline and passing this pipeline to `cross_val_score()` ensures that `StandardScaler` is fitted only on the four fitting folds. The fitted scaler is then used to transform both the fitting folds and the validation fold. This ensures that the validation fold is transformed using parameters learned elsewhere and the validation observations themselves do not help determine these parameters.

## 2. The confusion matrix

Accuracy tells us how many predictions were correct, but not **what kinds of mistakes were made**.

For binary classification, every prediction falls into one of four categories:

| |Actually positive|Actually negative|
|--|--|--|
|Predicted positive|True Positive (TP)|False Positive (FP)|
|Predicted negative|False Negative (FN)|True Negative (TN)|

The words **true/false** tell us whether the prediction was correct.

The words **poisitve/negative** tell us which class was predicted.

### Positive is something we define
But for our diagnostic interpretation today, we'll define: $$\boxed{\text{positive}=\text{malignant}}$$

With malignant as positive:
- **TP**: malignant patient correctly predicted malignant
- **FN**: malignant patient incorrectly predicted benign
- **FP**: benign patient incorrectly predicted as malignant
- **TN**: benign patient correctly predicted benign

Think about the consequences:

A **false negative** means:

> cancer is present, but the model says benign.

A **false positive** means:

> cancer is absent, but the model says malignant.

Those are very different errors, which is why accuracy alone can hide important information.

## 3. Precision, recall and specificity

### Recall/sensitivity
Recall asks:

> Of all the genuinely positive cases, how many did we detect?

$$\text{Recall}=\frac{TP}{TP+FN}$$

High recall means **few false negatives**.

### Precision
Precision asks:

> Of everything we predicted as positive, how much actually was positive?

$$\text{Precision}=\frac{TP}{TP+FP}$$

High precision means **few false positives**.

### Specificity
Specificity looks at the negative class:

> Of all genuinely negative cases, how many did we correctly identify as negative?

$$\text{Specificity}=\frac{TN}{TN + FP}$$

High specificity also corresponds to **few false positives**.


## 4. F1 score

Sometimes we want one metric that considers both precision and recall.

The F1 score is their harmonic mean:

$$F1=2\frac{\text{Precision}\times\text{Recall}}{\text{Precision}+\text{Recall}}$$

It becomes high only when **both precision and recall are reasonably high**.

F1 does not include true negatives directly, which is one reason it can be useful when the positive class is of particular interest or classes are imbalanced.

## 6. Conceptual check

1. A diagnostic classifier has extremely high recall for malignant tumours but relatively low precision. What kinds of errors would you expect it to make frequently, and what kind would it make rarely?

> With lower precision we would expect to see higher numbers of false positives, while with high recall we would expect to see fewer numbers of false negatives

2. Suppose missing a malignant tumour is considered substantially more harmful than incorrectly flagging a benign tumour for further investigation. Between precision and recall, which would you prioritise, and why?

> In this case, a false negative is more impactful than a false positive. While a false positive creates unneccessary stress for the falsely classified patient, we can assume it would lead to further investigation into the patient which would eventually determine that the tumour is in fact benign. On the other hand, a false negative, if not followed up, could lead to a patients malignant tumour being missed completley, which could cause substanitally more harm to the patient in the long run. For this reason, sensitivity should be prioritised for this scenario.

## 7. Out-of-fold predictions

In the previous sessions, `cross_val_score()` gave us five accuracy values.

But nonw we need something different: we need the individual predictions so we can count: $$TP, TN, FP, FN$$

We still don't want to touch `X_test`.

This is where **out-of-fold predictions** come in.

During 5-fold cross-validation: $$F_1,F_2,F_3,F_4,F_5$$

we fit on four folds and predict the remaining fold.

For example: $$F_2+F_3+F_4,F_5\rarr\text{model}\rarr\text{prediction for }F_1$$

Then repeat for every fold.

At the end, every observation in `X_train` has exactly one prediction, and crucially:

> that prediction came out from a model that was not fitted on that observation.

We combine all those predictions into one array:

$$ y_{\text{OOF}} $$

with the same length as y_train.

That lets us construct a confusion matrix and calculate classification metrics using the training data without simply evaluating a model on observations it was fitted on.

These are not predictions from one single fitted model. They come from the five temporary CV models.



## 8. Coding task — classification metrics

Use the scaled 5-NN pipeline from Session 3 and `X_train`, `y_train`.

1. Generate 5-fold out-of-fold predictions for every observation in X_train.
2. Construct the confusion matrix, explicitly treating the class order as [0, 1].
3. Extract:
    - $TP$
    - $FN$
    - $FP$
    - $TN$
4. Calculate:
    - accuracy;
    - precision;
    - recall/sensitivity;
    - specificity;
    - F1.
5. Report all five metrics.

In [ ]:
import numpy as np
from sklearn import metrics
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


breast_cancer = load_breast_cancer()

X = breast_cancer.data
y = breast_cancer.target

X_train, X_test, y_train, y_test = train_test_split(
  X,
  y, 
  test_size=0.2,
  random_state=42,
  stratify=y
)

# 5-nearest neighbours classifier with StandardScaler pipeline
neigh_scaled = Pipeline([
  ('scaler', StandardScaler()),
  ('classifier', KNeighborsClassifier(n_neighbors=5))
])

# Generate 5-fold OOF predictions for every observation in X_train
neigh_scaled_oof_pred = cross_val_predict(neigh_scaled, X_train, y_train, cv=5)

# Construct the confusion matrix
confusion_matrix = metrics.confusion_matrix(y_train, neigh_scaled_oof_pred, labels=[0, 1])

# Extract predictions
true_positives = confusion_matrix[0,0]
false_negatives = confusion_matrix[0,1]
false_positives = confusion_matrix[1,0]
true_negatives = confusion_matrix[1,1]

print('True positives:', true_positives)
print('False negatives:', false_negatives)
print('False positives:', false_positives)
print('True negatives:', true_negatives)

# Calculate metrics
accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
precision = true_positives / (true_positives + false_positives)
recall =  true_positives / (true_positives + false_negatives)
specificity = true_negatives / (true_negatives + false_positives)
F1 = 2 * (precision * recall) / (precision + recall)

# Note: Could have also used metrics.precision_score, metrics.recall_score, metrics.f1_score, 
# but since we already extracted confusion matrix values manual calculation seemed more intuitive

# Report all metrics
print('\nAccuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('Specificity:', specificity)
print('F1:', F1)

True positives: 160
False negatives: 10
False positives: 5
True negatives: 280

Accuracy: 0.967032967032967
Precision: 0.9696969696969697
Recall: 0.9411764705882353
Specificity: 0.9824561403508771
F1: 0.955223880597015


- How many malignant cases were missed?

> 10 malignant cases were missed

- How many benign cases were falsely flagged malignant?

> 5 cases were falsely flagged as malignant

- Is accuracy alone a sufficient description of this classifier?

> Accuracy alone is not sufficient because it does not distinguish between false positives and false negatives. This matters here because the two types of error have different consequences, even though the overall accuracy and F1 scores happen to be similar.

- Given our hypothetical assumption that false negatives are substantially more harmful, which metric would you nominate as the primary metric, and why?

> Recall should be prioritised as the primary metric, since it measures the proportion of actual malignant cases correctly detected. Maximising recall minimises the proportion of malignant cases missed as false negatives, which we have assumed are the more harmful error.